# Notebook 02: Data Cleaning & Feature Engineering

**Purpose:** Convert raw vehicle-level simulation log data into aggregated, clean, and ML/RL-ready features per road/lane segment.

### Mathematical Formulation of Derived Features
1. **Vehicle Count ($N_{r,t}$):** Total count of unique vehicles on road segment $r$ at time $t$.
2. **Average Speed ($ar{v}_{r,t}$):**
   $$ar{v}_{r,t} = rac{1}{N_{r,t}} \sum_{i=1}^{N_{r,t}} v_{i,t}$$
3. **Total CO2 ($C_{r,t}$):**
   $$C_{r,t} = \sum_{i=1}^{N_{r,t}} co2_{i,t}$$
4. **Average CO2 per Vehicle ($ar{c}_{r,t}$):**
   $$ar{c}_{r,t} = rac{C_{r,t}}{N_{r,t}}$$
5. **Total Waiting Time ($W_{r,t}$):**
   $$W_{r,t} = \sum_{i=1}^{N_{r,t}} wait_{i,t}$$
6. **Average Waiting Time per Vehicle ($ar{w}_{r,t}$):**
   $$ar{w}_{r,t} = rac{W_{r,t}}{N_{r,t}}$$
7. **Traffic Flow ($Q_{r,t}$):**
   $$Q_{r,t} = N_{r,t} \cdot ar{v}_{r,t}$$
8. **Congestion Indicator ($I_{r,t}$):**
   $$I_{r,t} = \begin{cases} 1 & \text{if } ar{v}_{r,t} < 5.0 \text{ m/s (18 km/h) and } N_{r,t} > 2 \\ 0 & \text{otherwise} \end{cases}$$
9. **Emission Intensity ($E_{r,t}$):**
   $$E_{r,t} = rac{C_{r,t}}{Q_{r,t} + \epsilon}$$
10. **Rolling CO2 ($C^{roll}_{r,t}$):** 5-step rolling window sum of road CO2.
11. **Rolling Waiting Time ($W^{roll}_{r,t}$):** 5-step rolling window sum of road waiting time.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path("../data/raw/simulation_data.csv")
rl_features_path = Path("../data/processed/rl_features.csv")


### Load and Prepare Data
We load a large sample (skiprows=every 5th row) to perform aggregating feature engineering across the entire simulation time.


In [ ]:
# Load every 5th row to have highly detailed spatial-temporal aggregations
df = pd.read_csv(raw_path, skiprows=lambda i: i > 0 and i % 5 != 0)
print(f"Loaded sample shape: {df.shape}")


### Perform Road-Level Aggregation

In [ ]:
# Sort by simulation time and vehicle id
df = df.sort_values(by=["simulation_time", "vehicle_id"]).reset_index(drop=True)

# Group by time and road
agg_funcs = {
    "vehicle_id": "count",
    "speed": "mean",
    "co2": ["sum", "mean"],
    "waiting_time": ["sum", "mean"],
    "nox": "sum",
    "fuel_consumption": "sum",
    "x": "mean",
    "y": "mean"
}

road_features = df.groupby(["simulation_time", "road_id"]).agg(agg_funcs).reset_index()

# Flatten columns
road_features.columns = [
    "simulation_time", "road_id", "vehicle_count", "average_speed",
    "total_CO2", "average_CO2_per_vehicle", "total_waiting_time", 
    "average_waiting_time", "total_nox", "total_fuel", "mean_x", "mean_y"
]

print(f"Aggregated road segments count: {road_features.shape[0]}")


### Engineer Derived Features

In [ ]:
# 1. Traffic Flow
road_features["traffic_flow"] = road_features["vehicle_count"] * road_features["average_speed"]

# 2. Congestion Indicator
road_features["congestion_indicator"] = np.where(
    (road_features["average_speed"] < 5.0) & (road_features["vehicle_count"] > 2), 1, 0
)

# 3. Emission Intensity
road_features["emission_intensity"] = road_features["total_CO2"] / (road_features["traffic_flow"] + 1e-5)

# 4. Rolling Temporal Features per Road Segment
road_features = road_features.sort_values(by=["road_id", "simulation_time"]).reset_index(drop=True)

road_features["rolling_CO2"] = road_features.groupby("road_id")["total_CO2"].transform(
    lambda x: x.rolling(window=5, min_periods=1).sum()
)

road_features["rolling_waiting_time"] = road_features.groupby("road_id")["total_waiting_time"].transform(
    lambda x: x.rolling(window=5, min_periods=1).sum()
)

# 5. Vehicle Density (using vehicle count as a proxy for road density since length is not direct)
road_features["vehicle_density"] = road_features["vehicle_count"]

print(road_features.head())


### Save RL Features

In [ ]:
# Sort back by simulation_time
road_features = road_features.sort_values(by=["simulation_time", "road_id"]).reset_index(drop=True)
road_features.to_csv(rl_features_path, index=False)
print(f"RL features successfully saved to: {rl_features_path}")
